# Buffs and Effects

This notebook shows how to:
1. Extract buffs from items
2. Use the `.buff_ui` property for automatic formatting
3. Navigate buff structures
4. Find what buildings are affected

In [1]:
from assetextractor.extraction.utils import Config
from assetextractor.parsing.core.assets import AssetCache
from assetextractor.parsing.core.texts import StandardTextConverter

# Load assets
config = Config.from_json("config.json")
assets = AssetCache.load(config)
texts = assets.texts

# Set language
LANGUAGE = "english"
texts.converter = StandardTextConverter(LANGUAGE)

print("Assets loaded!")

Assets loaded!


## Get an Item with Buffs

Let's examine a specialist item that provides buffs:

In [2]:
# Get Zorsines, Sarmatian Swordshaper (a legendary item)
item = assets.get(80510)

name = item.text() if item.text else 'N/A'
print(f"Item: {name}")
print(f"GUID: {item.guid}")
print(f"Template: {item.template.name}")

Item: Zorsines, Sarmatian Swordshaper
GUID: 80510
Template: ItemWithBoost


## Use .buff_ui Property (Easiest Method)

The `.buff_ui` property automatically extracts and formats all buffs:

In [3]:
# Get all BuffUI objects from the item
buff_list = item.buff_ui

print(f"\nFound {len(buff_list)} buffs:\n")

for i, buff_ui in enumerate(buff_list, 1):
    # BuffUI has: icon, text, value, literal
    text_str = buff_ui.text.values.get(LANGUAGE) if hasattr(buff_ui.text, 'values') else str(buff_ui.text)
    
    print(f"{i}. {text_str}")
    if buff_ui.value:
        print(f"   Value: {buff_ui.value}")
    if buff_ui.icon:
        print(f"   Icon: {buff_ui.icon.value.stem if buff_ui.icon.value else 'N/A'}")
    print()


Found 2 buffs:

1. Productivity
   Value: +30%
   Icon: icon_2d_productivity_0

2. Fuel efficiency
   Value: +25%
   Icon: icon_3d_charcoal_goods_0



## Manual Buff Extraction

For more control, navigate the buff structure manually:

In [4]:
# Items have Effect.Buffs that reference BuildingBuff or ShipBuff assets
if hasattr(item, 'Effect'):
    buffs = item.Effect.Buffs
    
    print(f"Item has {len(buffs)} buff references:\n")
    
    for i, buff_entry in enumerate(buffs, 1):
        # Get the referenced buff asset
        buff_asset = buff_entry.GUID()
        
        if buff_asset:
            buff_name = buff_asset.text() if buff_asset.text else 'N/A'
            print(f"{i}. Buff: {buff_name} (GUID: {buff_asset.guid})")
            print(f"   Template: {buff_asset.template.name}")
            
            # Show a few buff properties
            if hasattr(buff_asset, 'FactoryUpgrade'):
                productivity = buff_asset.find("FactoryUpgrade.ProductivityUpgrade")
                if productivity and productivity():
                    print(f"   Productivity: +{productivity()}%")
            
            print()

Item has 1 buff references:

1. Buff: Zorsines, Sarmatian Swordshaper (GUID: 80511)
   Template: BuildingBuff
   Productivity: +30.0%



## Get Affected Buildings (Targets)

Items specify which buildings they affect:

In [5]:
if hasattr(item, 'Effect'):
    targets = item.Effect.Targets
    
    if targets and len(targets) > 0:
        print(f"This item affects {len(targets)} building type(s):\n")
        
        for i, target_entry in enumerate(targets, 1):
            target_asset = target_entry.GUID()
            if target_asset:
                target_name = target_asset.text.values.get(LANGUAGE, 'N/A') if target_asset.text else 'N/A'
                print(f"{i}. {target_name} (GUID: {target_asset.guid})")
    else:
        print("No specific targets (applies to all compatible buildings)")

This item affects 1 building type(s):

1. Armouries (GUID: 50614)


## Compare Different Buff Types

Let's look at different types of buffs:

In [6]:
# Example items with different buff types
example_items = [
    80510,  # Zorsines (Productivity + Additional Output)
    51283,  # Casponia Casta (Additional Output)
]

for item_guid in example_items:
    item = assets.get(item_guid)
    if not item:
        continue
    
    name = item.text.values.get(LANGUAGE, 'N/A') if item.text else 'N/A'
    print(f"\n{'='*60}")
    print(f"{name} (GUID: {item_guid})")
    print('='*60)
    
    # Use buff_ui for clean output
    buff_list = item.buff_ui
    
    for buff_ui in buff_list:
        # String representation is already formatted
        print(f"  • {buff_ui}")


Zorsines, Sarmatian Swordshaper (GUID: 80510)
  • icon_2d_productivity_0 | Productivity | +30%
  • icon_3d_charcoal_goods_0 | Fuel efficiency | +25%

Casponia Casta, Sacerdos Cereris (GUID: 51283)
  • [icon_2d_religion_0 | Belief Area Effect | +1]
  • icon_2d_productivity_0 | Productivity | +25%


## Iterate All Items and Count Buff Types

Let's analyze what types of buffs are most common:

In [7]:
from collections import Counter

buff_template_count = Counter()

# Iterate through all items
items_template = assets.templates["Item"]

for item in items_template.assets:  # First 100 items
    if hasattr(item, 'Effect'):
        buffs = item.Effect.Buffs
        for buff_entry in buffs:
            buff_asset = buff_entry.GUID()
            if buff_asset:
                buff_template_count[buff_asset.template.name] += 1

print("Most common buff templates:\n")
for template_name, count in buff_template_count.most_common(10):
    print(f"  {template_name}: {count}")

Most common buff templates:

  BuildingBuff: 260
  ShipBuff: 47
  DefenseBuildingBuff: 9
  AreaBuff: 4


## Next Steps

- `05_construction_materials.ipynb` - Analyze building costs
- `06_backtrack_effects.ipynb` - Find which items provide specific buffs
- `07_area_effects.ipynb` - Work with area/radius effects